# 03 - Train the Classification Head (derived 3-class labels)

Trains DenseNet-121 encoder + classification head on a **combined** dataset built from both SIIM-ACR and RSNA: every image gets a label

- `0` = normal (no pneumothorax, no pneumonia)
- `1` = pneumothorax (from SIIM-ACR positive masks)
- `2` = pneumonia (from RSNA positive boxes)

This mirrors how the original MultiCheXNet paper derives its 3-way classification labels straight from the segmentation/detection ground truth - no separate classification dataset needed.

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.siim_acr import (SIIMACRDataset, build_siim_dataframe,
                                    train_val_split as seg_split,
                                    get_default_train_augmentation as seg_aug)
from src.datasets.rsna import (RSNADataset, load_rsna_dataframe,
                                train_val_split as det_split,
                                get_default_train_augmentation as det_aug)
from src.utils.visualize import show_training_curves

print("device:", config.DEVICE)


## 1. Paths (same as notebooks 01 / 02)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


seg_df = build_siim_dataframe(SEG_CSV_PATH, SEG_IMG_PATH)
det_df = load_rsna_dataframe(DET_CSV_PATH)

seg_train_ids, seg_val_ids = seg_split(seg_df, val_fraction=0.15)
det_train_ids, det_val_ids = det_split(det_df, val_fraction=0.15)


## 2. Small wrappers that expose only `(image, label)`

In [ ]:
class LabelOnly(Dataset):
    """Wraps a dataset whose __getitem__ returns (image, ..., label, ...) and
    exposes only (image, label)."""
    def __init__(self, base_dataset, label_index):
        self.base = base_dataset
        self.label_index = label_index

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        item = self.base[idx]
        return item[0], item[self.label_index]

# Previous run had no augmentation at all on classifier training data,
# which is a big reason it overfit so fast (train_acc 79%->93%, val_acc
# peaked at epoch 2 then degraded). Add the same augmentations used to
# train the segmenter/detector heads.
seg_train_ds = LabelOnly(SIIMACRDataset(seg_df, seg_train_ids, augmentation=seg_aug()), label_index=2)
seg_val_ds   = LabelOnly(SIIMACRDataset(seg_df, seg_val_ids), label_index=2)
det_train_ds = LabelOnly(RSNADataset(det_df, det_train_ids, DET_IMG_PATH, augmentation=det_aug()), label_index=2)
det_val_ds   = LabelOnly(RSNADataset(det_df, det_val_ids, DET_IMG_PATH), label_index=2)

train_ds = ConcatDataset([seg_train_ds, det_train_ds])
val_ds = ConcatDataset([seg_val_ds, det_val_ds])
print(len(train_ds), "train images,", len(val_ds), "val images")

BATCH_SIZE = 16
# num_workers=2 for throughput (DICOM decode + resize is CPU-bound). You may
# see a harmless "AssertionError: can only test a child process" during
# worker cleanup under Kaggle/Jupyter - cosmetic only, doesn't affect
# training.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          persistent_workers=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                        persistent_workers=True)


## 3. Build the model (classification head only) and train

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=True,
                     add_detector=False, add_segmenter=False).to(config.DEVICE)

# weight_decay adds L2 regularization - previous run had none, and was the
# fastest of the three heads to overfit (val_acc peaked at epoch 2/10).
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1)

# BUG FIX: checkpoint used to be saved to "/content/classifier_best.pt"
# (a leftover Colab path) instead of the actual Kaggle working directory,
# which is why notebook 04's load_partial() raised
# FileNotFoundError: .../classifier_best.pt. Fixed to match the path
# notebook 04 actually looks for.
CHECKPOINT_PATH = "/kaggle/working/multi-task-chest-xray-analysis-system/classifier_best.pt"

EPOCHS = 10
history = engine.fit(
    engine.train_one_epoch_classifier, engine.evaluate_classifier,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path=CHECKPOINT_PATH,
    monitor="acc", mode="max", scaler=scaler,
    scheduler=scheduler, patience=4,
)


In [ ]:
show_training_curves({k: v for k, v in history.items() if 'loss' in k})
show_training_curves({k: v for k, v in history.items() if 'acc' in k})


Checkpoint saved to `/content/classifier_best.pt`. Reused in `04_Train_MTL_Joint.ipynb`.